# jetson/pilot_deitb_ext_export.py -- Colab runner

DeiT-B CROSSOVER CHARACTERIZATION, step 1 -- see `jetson/README.md`'s
"DeiT-B crossover characterization" section. Fills the gap between the
DeiT-B pilot's two buckets (N=197 NO-GO, N=69 BORDERLINE) with N=133
and N=101, around the naive ~N=87 crossover estimate. Runs on Colab's
GPU -- deliberately NOT the Jetson (same `nvidia-modelopt` blocker as
every prior QDQ export -- see `jetson/README.md`'s "Batch-8 INT8
pilot" section).

This repo is public -- cloning it needs no authentication.
**HF auth** -- add a Colab Secret named `HF_TOKEN`, needed to stream
128 calibration images (this run needs its OWN set, same reason as the
original DeiT-B pilot -- cannot reuse any prior calibration capture).
Never a literal in any cell.

Runtime: Runtime -> Change runtime type -> any GPU (T4 is enough).

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
get_ipython().system('nvidia-smi')  # noqa: F821

CUDA available: True
GPU: Tesla T4
Thu Jul 30 20:19:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

In [2]:
import os

REPO_URL = "https://github.com/fawadsarim/tandem.git"
REPO_DIR = "/content/tandem"

if not os.path.exists(REPO_DIR):
    get_ipython().system(f'git clone -q {REPO_URL} {REPO_DIR}')  # noqa: F821
else:
    print(f"{REPO_DIR} already exists -- pulling latest instead of re-cloning.")
    get_ipython().system(f'git -C {REPO_DIR} pull -q')  # noqa: F821

get_ipython().run_line_magic('cd', REPO_DIR)  # noqa: F821

/content/tandem


## Isolated venv for `requirements-host.txt` + `nvidia-modelopt`

In [3]:
get_ipython().system('pip install -q virtualenv')  # noqa: F821
get_ipython().system('python3 -m virtualenv -q --clear /content/venv_jetson_host')  # noqa: F821
get_ipython().system('/content/venv_jetson_host/bin/pip install -q -r jetson/requirements-host.txt')  # noqa: F821
get_ipython().system('/content/venv_jetson_host/bin/pip install -q nvidia-modelopt requests "numpy==1.26.4" "pandas==2.2.2"')  # noqa: F821
VENV_PYTHON = "/content/venv_jetson_host/bin/python"

import subprocess  # noqa: E402

_check = subprocess.run(
    [VENV_PYTHON, "-c",
     "import torch, numpy, pandas, modelopt.torch.quantization; "
     "print('TANDEM_ENV_CHECK', torch.__version__, torch.cuda.is_available(), numpy.__version__, pandas.__version__)"],
    capture_output=True, text=True,
)
print(_check.stdout)
if _check.returncode != 0:
    print(_check.stderr[-3000:])
_result_lines = [l for l in _check.stdout.splitlines() if l.startswith("TANDEM_ENV_CHECK")]
assert _result_lines, f"venv import check failed -- see stderr above.\n{_check.stderr[-3000:]}"
_, _torch_version, _cuda_ok, _numpy_version, _pandas_version = _result_lines[-1].split()
assert _cuda_ok == "True", (
    f"venv torch {_torch_version} cannot see the GPU -- Runtime -> Change runtime type -> a GPU runtime."
)
print(f"torch {_torch_version}, numpy {_numpy_version}, pandas {_pandas_version}, "
      f"CUDA available: True, modelopt.torch.quantization importable: True")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 39.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
TANDEM_ENV_CHECK 2.11.0+cu130 True 1.26.4 2.2.2

torch 2.11.0+cu130, numpy 1.26.4, pandas 2.2.2, CUDA available: True, modelopt.torch.quantization importable: True


## `datasets`/`huggingface_hub` -- DEFAULT Colab kernel, not the venv

In [4]:
get_ipython().system('pip install -q datasets huggingface_hub')  # noqa: F821

## HF auth (env var only, never a literal)

In [5]:
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN Colab Secret -- see markdown above."
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

## Materialize this run's OWN 128-image calibration set

In [6]:
import pathlib  # noqa: E402
from datasets import load_dataset  # noqa: E402

CALIB_DIR = pathlib.Path("/content/calib_imagenet_deitb_ext/class_0")
CALIB_DIR.mkdir(parents=True, exist_ok=True)
N_NEEDED = 128

existing = len(list(CALIB_DIR.glob("*.jpg")))
if existing >= N_NEEDED:
    print(f"{existing} calibration images already present -- skipping re-download.")
else:
    ds = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True, token=HF_TOKEN)
    n_written = 0
    for i, example in enumerate(ds):
        if n_written >= N_NEEDED:
            break
        img = example["image"].convert("RGB")
        img.save(CALIB_DIR / f"img_{n_written:04d}.jpg", quality=95)
        n_written += 1
    print(f"Wrote {n_written} calibration images to {CALIB_DIR.parent}")

README.md:   0%|          | 0.00/87.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Wrote 128 calibration images to /content/calib_imagenet_deitb_ext


## Run pilot_deitb_ext_export.py

In [7]:
_proc = subprocess.Popen(
    [VENV_PYTHON, "-u", "jetson/pilot_deitb_ext_export.py", "--imagenet-dir", str(CALIB_DIR.parent)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for _line in _proc.stdout:
    print(_line, end="")
_proc.wait()
assert _proc.returncode == 0, f"pilot_deitb_ext_export.py failed (exit code {_proc.returncode}) -- see the real error printed above."

=== pilot_deitb_ext_export.py: deit_base_patch16_224, FP + explicit-QDQ INT8, batch=1, buckets=(133, 101) (crossover characterization) ===
Loaded deit_base_patch16_224.fb_in1k, checkpoint_hash=d0d329d779597c2bec332181ea0ab920d55a7113b0aef8c0e1ea653744a2a559

--- Calibration capture (this run's own -- see module docstring) ---
Captured 128 images, calib_hash=ed0ce9377877e36363697e4aeac1f9cb64f3c1e49bd6f23ee5e99741a430885b

--- bucket n_tokens=133 ---
/content/venv_jetson_host/lib/python3.12/site-packages/timm/layers/attention.py:31: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if is_causal:
  FP:  /content/tandem/jetson/out/pilot_deitb_ext/block_seg_n133.onnx  max_abs_diff=1.669e-05  OK
  Loaded 128 real calibration activations from /content/tandem/jetson/out/pilot_dei

## Results

In [8]:
import json  # noqa: E402

manifest = json.loads(pathlib.Path("jetson/out/pilot_deitb_ext/pilot_deitb_ext_export_manifest.json").read_text())
print(json.dumps(manifest, indent=2))

{
  "model_name": "deit_base_patch16_224",
  "resolved_pretrained_tag": "deit_base_patch16_224.fb_in1k",
  "checkpoint_state_hash": "d0d329d779597c2bec332181ea0ab920d55a7113b0aef8c0e1ea653744a2a559",
  "d_model": 768,
  "n_blocks": 12,
  "n_tokens_total": 197,
  "buckets": [
    133,
    101
  ],
  "calibration": {
    "n_calib_images": 128,
    "calib_hash": "ed0ce9377877e36363697e4aeac1f9cb64f3c1e49bd6f23ee5e99741a430885b"
  },
  "max_abs_diff_tol": 0.0001,
  "results": {
    "133": {
      "fp_onnx_path": "/content/tandem/jetson/out/pilot_deitb_ext/block_seg_n133.onnx",
      "fp_max_abs_diff": 1.6689300537109375e-05,
      "qdq_onnx_path": "/content/tandem/jetson/out/pilot_deitb_ext/block_seg_n133_qdq.onnx",
      "qdq_ops": 16
    },
    "101": {
      "fp_onnx_path": "/content/tandem/jetson/out/pilot_deitb_ext/block_seg_n101.onnx",
      "fp_max_abs_diff": 1.430511474609375e-05,
      "qdq_onnx_path": "/content/tandem/jetson/out/pilot_deitb_ext/block_seg_n101_qdq.onnx",
      "qd

## Get the files onto your laptop (then onto the Jetson)

In [9]:
get_ipython().system('cd jetson/out && zip -qr /content/pilot_deitb_ext_onnx.zip pilot_deitb_ext --exclude "*/calib/*"')  # noqa: F821
from google.colab import files  # noqa: E402
files.download("/content/pilot_deitb_ext_onnx.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>